#Assignment 5: Transformer Encoders for Text Classification [100 points total]

**DUE DATE: NOVEMBER 14, 2025**

Text classification is the problem of assigning a class label (a single output) to a sequence of text tokens (many inputs).

In this assignment, you will implement and train a transformer encoder model to perform binary classification in this manner.

**To submit: this `.ipynb` file with your modifications and output.**

**There are conceptual questions at the end - be sure to answer these as well.**

**DO NOT CLEAR OUTPUTS!**

## Dataset

The IMDB dataset consists of 25K positive and 25K negative movie reviews for training, and a further set of 25K positive and 25K negative reviews for testing (a total of 100K reviews), all sourced from [IMDB](https://www.imdb.com/). The training and testing sets are, of course, disjoint.

It was produced by NLP researchers at Stanford, the goal being to train intelligent text processing systems to recognize written opinions as positive or negative (sometimes referred to as "[sentiment analysis](https://en.wikipedia.org/wiki/Sentiment_analysis)"), one of the earliest and best-studied NLP tasks.

The dataset and its metadata can be accessed via HuggingFace Datasets [here](https://huggingface.co/datasets/stanfordnlp/imdb).

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset
imdb = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Below is a wrapper class that subclasses [`torch.utils.data.Dataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset). The point of doing this is to later further wrap our IMDBDataset inside of a [`torch.utils.data.DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader), which has useful functionalities for quickly loading batches of data.

In [3]:
from torch.utils.data import Dataset

class IMDBDataset(Dataset):

    # any subclass of Dataset must have __init__(self, ...)
    def __init__(self, split='train'):

        # inherit all functionality from Dataset
        super().__init__()

        # check that there is one label for every review
        assert len(imdb[split]['text']) == len(imdb[split]['label'])

        # data structure of our choosing
        self.pairs = list(zip(imdb[split]['text'], imdb[split]['label']))

    # any subclass of Dataset must have __len__(self)
    # x.__len__() is the same as len(x)
    def __len__(self):
        return len(self.pairs)

    # any subclass of Dataset must have __getitem__(self, idx)
    # x.__getitem__(i) is the same as x[i]
    # be careful when writing this method:
    # be sure it accommodates slices! x[i:j]
    def __getitem__(self, idx):
        return self.pairs[idx]

In [4]:
train_set = IMDBDataset('train')
test_set = IMDBDataset('test')

## Dataloaders

A [`torch.utils.data.DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) is a convenient wrapper for a dataset. You can iterate over it to access batches of data.
- `dataset` is the dataset you want to wrap.
- `batch_size` is the batch size.
- `shuffle=True` will randomly shuffle the data each time you iterate over the `DataLoader`.
- `num_workers` will spawn additional processes on CPU to load up batches more efficiently.

In [20]:
import torch
from torch.utils.data import DataLoader
batch_size = 32

OVERFIT = False # test on the train set to check whether the model is learning
NUM_SAMPLES = 40_000 # number of train samples, also number of test samples

train_loader = DataLoader(
    dataset=Subset(
        train_set,
        indices=torch.randint(0, len(train_set), (NUM_SAMPLES,))
    ),
    batch_size=batch_size,
    shuffle=True,
    num_workers=1
)

if OVERFIT: # test on the train set to check whether the model is learning
    test_loader = train_loader
else:
    test_loader = DataLoader(
        dataset=Subset(
            test_set,
            indices=torch.randint(0, len(test_set), (NUM_SAMPLES,))
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=1
    )

### A brief note on data size

You may find that it is very difficult to get the model to learn the entire training set, as it's very large. Consider adjusting `NUM_SAMPLES` larger and larger as you experiment.

Try to get the model to learn a few hundred examples at first, and evaluate on the same amount of testing data. Then, you can increase to a larger number of samples.

You can set `OVERFIT = True` below to test on the training set to confirm the model is effectively learning its training data. In principle, it should not take terribly much training to get the model to learn its training set to 100% accuracy.

## Model

### Imports for Transformer Encoder Architecture

- Neural network building blocks: [`torch.nn`](https://pytorch.org/docs/stable/nn.html)
- NN functions: [`torch.nn.functional`](https://pytorch.org/docs/stable/nn.functional.html)

In [7]:
import torch
from torch import nn
import torch.nn.functional as F

### Transformer Encoder architecture

Similar to the models you developed in previous assignments, we must:
- subclass [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
- define an `__init__(self, ...)` method
    - Define an [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) layer
    - Define a fixed (frozen) positional encoding (use [`self.register_buffer()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer)). Take a look at [this blog post](https://jalammar.github.io/illustrated-transformer/), section "Representing The Order of The Sequence Using Positional Encoding" for a conceptual understanding of what this does. There are fairly standard ways to do this in Torch. See if you can find an implementation of this online.
    - Define an [`nn.TransformerEncoder`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)
    - Define a classification head using an [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- define a `forward(self, ...)` method
    - Embed the input text tokens from the tokenizer
    - Pass the embedded sequence and the source key padding mask to the transformer
    - Pass the transformer's output to the classification head
    - Return the logits

***TODO: Complete the `__init__()` [25 points] and `forward()` [25 points] methods of the following SentimentTransformer class.***

In [14]:
import math
class SentimentTransformer(nn.Module):
    def __init__(
        self,
        tokenizer,
        max_length,
        d_model,
        nhead,
        num_layers,
        dim_feedforward,
        dropout,
        activation
    ):
        super().__init__()

        self.max_length = max_length
        self.d_model = d_model
        self.tokenizer = tokenizer
        self.vocab_size = len(self.tokenizer)

        # TODO: embedding table
        self.embedding = nn.Embedding(self.vocab_size, d_model, padding_idx=0)
        # TODO: sinusoidal positional encoding
        pos_encoding = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pos_encoding[:, 0::2] = torch.sin(position * div_term)
        pos_encoding[:, 1::2] = torch.cos(position * div_term)
        pos_encoding = pos_encoding.unsqueeze(0) # Add a batch dimension: (1, max_length, d_model)
        # Register 'pos_encoding' as a buffer, so it's part of the model state but not trained as a parameter.
        self.register_buffer('pos_encoding', pos_encoding)
        self.dropout = nn.Dropout(p=dropout)
        # TODO: transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True
        )
        # Stacking the encoder layers
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        # TODO: binary classification projection layer (classification head)
        self.classification_head = nn.Linear(d_model, 1)

    def forward(self, src):
        # TODO: embed src
        # remember, src has 'input_ids' and 'attention_mask'
        # Extract input_ids and attention_mask
        input_ids = src['input_ids']
        attention_mask = src['attention_mask']
        batch_size, seq_len = input_ids.shape
        # Check if seq_len exceeds max_length
        if seq_len > self.max_length:
            raise ValueError(
                f"Input sequence length ({seq_len}) exceeds model's "
                f"max_length ({self.max_length})"
            )
        # Scale the embedding by sqrt(d_model) as per the Transformer paper
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        # TODO: add positional encodings to every position
        # Slicing self.pe to match the input's seq_len
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x)

        # TODO: mask out padding, and cast to bool again, src has 'input_ids' and 'attention_mask'
        padding_mask = (attention_mask == 0)
        # TODO: pass through encoder
        encoder_output = self.transformer_encoder(
            src=x,
            src_key_padding_mask=padding_mask
        )

        # TODO: select 1st output embedding, flatten, or make some other choice first token (often [CLS] token)
        cls_embedding = encoder_output[:, 0, :]
        # TODO: pass through final feedforward network (classification head)
        logits = self.classification_head(cls_embedding)
        # TODO: output logits of the right shape for BCEWithLogitsLoss
        # Squeeze the last dimension
        return logits.squeeze(-1)

### Tokenizer

Training a tokenizer yourself is not too painful, but an even easier choice is to load up a pretrained one such as the one used by [BERT](https://bibbase.org/service/mendeley/bfbbf840-4c42-3914-a463-19024f50b30c/file/6375d223-e085-74b3-392f-f3fed829cd72/Devlin_et_al___2019___BERT_Pre_training_of_Deep_Bidirectional_Transform.pdf.pdf). If you're curious about training your own, look into [SentencePiece](https://github.com/google/sentencepiece).

Otherwise, let's use BERT's tokenizer, available on [HuggingFace](https://huggingface.co/google-bert/bert-base-uncased):

In [15]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# maximum length of sequences we will process
# longer sequences will be truncated to this length
# shorter sequences will be padded to this length
max_length = 128

# take a look at how it splits up words
# note that the number of integer IDs is longer than the number of subwords
# this is because BERT adds a "start" and "end" symbol
string = 'This is a short sentence containing the word "carboniferous."'
print('\ncalling tokenizer.tokenize:')
tokens = tokenizer.tokenize(string)
print(tokens, f'(len = {len(tokens)})')
print('\ncalling tokenizer.encode:')
encoded = tokenizer.encode(string)
print(encoded, f'(len = {len(encoded)})')

# tokenizer use, can also just say tokenizer(...) rather than tokenizer.__call__(...)
tokens = tokenizer.__call__(
    string, # text to tokenize
    padding='max_length', # pad sequences to the max_length if necessary
    truncation=True, # truncate long sequences if necessary
    max_length=max_length, # truncate and/or pad to this length
    return_tensors='pt' # give torch.Tensor as the output of tokenizing
)['input_ids'] # there's also two other things in the encoding, we just want the ids

# take a look at the input ids
print('\ncalling the tokenizer with max length padding:')
print(tokens)

# how many tokens in the model's vocabulary?
print('tokenizer\'s vocab size:', tokenizer.vocab_size, '=', len(tokenizer))


calling tokenizer.tokenize:
['this', 'is', 'a', 'short', 'sentence', 'containing', 'the', 'word', '"', 'carbon', '##iferous', '.', '"'] (len = 13)

calling tokenizer.encode:
[101, 2023, 2003, 1037, 2460, 6251, 4820, 1996, 2773, 1000, 6351, 23930, 1012, 1000, 102] (len = 15)

calling the tokenizer with max length padding:
tensor([[  101,  2023,  2003,  1037,  2460,  6251,  4820,  1996,  2773,  1000,
          6351, 23930,  1012,  1000,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0, 

#### A brief note on why we need padding and truncation

Although transformers can handle variable-length sequences, they cannot process a batch of sequences of different lengths all at once.

Consider a batch like:

$$ [0, 4, 8, 9, 7] \\ [0, 2] \\ [0, 3, 5] $$

Let's say we (somehow) have processed, in parallel, the first two tokens in each sequence using linear algebra operations on the hardware. We still have to process:

$$ [8, 9, 7] \\ [] \\ [5] $$

A generic algorithm for handling matrices and vectors is going to throw an error when it sees that there are empty entries here - batched inputs cannot be "ragged" in this way. We require a tensor to have well-defined dimensions.

One solution is to "pad" inputs to either (a) the length of the longest input sequence or (b) a pre-specified maximum length. Let's assume our padding token is index $1$.

Option (a):

$$ [0, 4, 8, 9, 7] \\ [0, 2, 1, 1, 1] \\ [0, 3, 5, 1, 1] $$

Option (b), with maximum length $8$:

$$ [0, 4, 8, 9, 7, 1, 1, 1] \\ [0, 2, 1, 1, 1, 1, 1, 1] \\ [0, 3, 5, 1, 1, 1, 1, 1] $$

On the other hand, truncation allows us to specify a maximum length to pad sequences to, while also handling cases where a sequence is longer than that maximum length. We simply "cut off" the sequence beyond that maximum length (this sequence won't actually need any padding).

Truncation with maximum length $2$:

$$ [0, 4] \\ [0, 2] \\ [0, 3] $$

Truncation with maximum length $4$ and padding to maximum length:

$$ [0, 4, 8, 9] \\ [0, 2, 1, 1] \\ [0, 3, 5, 1] $$

## Training and evaluation

### Imports for training and evaluation

- [`torch.optim.Adam`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html), [`torch.optim.AdamW`](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
- [`torch.nn.BCEWithLogitsLoss`](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [`torch.utils.data.DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
- [`torch.utils.data.Subset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset)
- [`time.time`](https://docs.python.org/3/library/time.html#time.time)
- [`sklearn.metrics.classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

In [16]:
from torch.optim import Adam, AdamW # optimization algorithm
from torch.nn import BCEWithLogitsLoss # loss function
from torch.utils.data import DataLoader # iterates over dataset in batches
from torch.utils.data import Subset # subsets a dataset
from time import time # time things
from sklearn.metrics import classification_report # evaluation metrics

### Hyperparameters

Hyperparameters are things chosen by the programmer, as opposed to parameters, which are optimized by a learning algorithm.

Some hyperparameters we are setting:
- **Tokenizer** (`tokenizer`) - we will just use BERT's for convenience. The tokenizer determines the vocabulary size (number of unique tokens), and whether any control symbols (start-of-sequence token, end-of-sequence token, padding token) are used.
- **Maximum length** (`max_length`) - the longest sequence length we will process. A longer sequence length takes longer to process, but captures more of the information in the sequence. There can also be more wasted compute due to excess padding.
- **Model dimension** (`d_model`), a choice. It's harder to train/slower/prone to overfitting a larger embedding, but a larger embedding may perform better.
- **Feed-forward size** (`dim_feedforward`), also a choice with the same issues. It's often 2-4x the model dimension.
- **Number of layers** (`n_layers`), also a choice with the same issues. A deeper network can capture more complex phenomena that a shallower network cannot.
- **Dropout probability** (`dropout`) - a little dropout often helps regularize a model and prevents overfitting.
- **Batch size** (`batch_size`) - how many inputs to process in parallel, and also, how many inputs to process before updating the model's parameters. A smaller batch size may cause overfitting (over-reliance on just a few examples to update the model), but uses less memory.
- **Epochs** (`epochs`) - how many times you want the model to see the training set. More epochs will train the model better, but can cause overfitting.
- **Optimization algorithm** (`optimizer`) - [`Adam`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) and [`AdamW`](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) are popular choices, but other choices may do better.
- **Learning rate (`lr`) - THE MOST IMPORTANT HYPERPARAMETER** - this has been [shown empirically](https://proceedings.mlr.press/v28/bergstra13.html). Typical choices are orders of magnitude smaller than 1 (1e-2, 1e-3, 1e-4...).
- **Loss function** (`criterion`) - kind of a hyperparameter, but really, it determines what you are teaching the model. Binary cross-entropy loss is appropriate for binary classification. BCE will force outputs for the positive class to be larger (greater than 0), and outputs for the negative class to be smaller (less than 0). The [`BCEWithLogitsLoss`](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) is a useful implementation that includes the sigmoid function - i.e. there is no need to normalize your output, just pass the raw logits and labels to the loss function.

***TODO: Adjust the hyperparameters to train the model to at least 75% accuracy [10 points]. NOTE: You should do this part last. You may adjust anything you want, though LR and epochs must be adjusted.***

In [17]:
# check if GPU is available, and let device be 'cuda' if so, otherwise 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# our model
trf = SentimentTransformer(
    tokenizer,
    max_length,
    d_model=128,
    nhead=4,
    num_layers=4,
    dim_feedforward=256,
    dropout=0,#0.05,
    activation="relu"
).to(device) # move model onto GPU if available

# number of examples to process in parallel
# we will perform only one optimization step per batch
batch_size = 32

# number of times the model sees the training data
epochs = 5 # TODO: select a better number of epochs

# Adam is the most beloved optimizer, 'lr' is the learning rate
optimizer = AdamW(
    trf.parameters(),
    lr=0.001, # TODO: select a better learning rate
    weight_decay=1e-1
)
# loss function for binary classification
criterion = BCEWithLogitsLoss()

### Testing a forward pass to make sure shapes are compatible

We are picking random tensors for `src`, `mask`, and `labels` to make sure that the model functions correctly.

While these tensors are randomly generated, they will "look" like real inputs.

For clarity, the `src` tensor:
- `src`'s entries range between $0$ and the vocab size, which is the valid range of input token IDs for the model.
- `src` has shape `(batch_size, max_length)`, simulating a batch of `batch_size` sequences padded/truncated to length `max_length`.
- `src` has datatype `torch.int64`, which is the kind of tensor our tokenizer will produce.
- `src` is on the GPU if it's available.

The `mask` tensor:
- `mask` has binary entries ($0$ or $1$).
- `mask` has the same shape as `src` - every token is either masked ($0$) or not masked ($1$).
- `mask` has datatype `torch.int64`, which is easily converted to boolean with `.bool()` in the model's forward pass.
- `mask` is on the GPU if it's available.

The `labels` tensor:
- `labels` has binary entries ($0$ or $1$).
- `labels` has the shape `(batch_size,)` because we only have one label for each sequence in the batch ($0$ or $1$).
- `labels` has datatype `torch.float32`, which is necessary for the loss computation.
- `labels` is on the GPU if it's available.

In [18]:
# test the model's forward() method

src = torch.randint( # random valid input
    0,
    tokenizer.vocab_size,
    (batch_size, max_length),
    dtype=torch.int64,
    device=device
)

mask = torch.randint( # random boolean mask
    0,
    2,
    (batch_size, max_length),
    dtype=torch.int64,
    device=device
)

out = trf({ # transformer will expect a dictionary like this
    'input_ids': src,
    'attention_mask': mask
})

labels = torch.randint( # random valid labels, binary classification
    0,
    2,
    (batch_size,),
    dtype=torch.float32,
    device=device
)

loss = criterion(out, labels)

print('src', src)
print(src.shape, '\n')
print('mask', mask)
print(mask.shape, '\n')
print('out', out)
print(out.shape, '\n')
print('labels', labels)
print(labels.shape, '\n')
print('loss', loss)

src tensor([[16402, 18971,  5488,  ..., 28703, 12735, 26794],
        [28044,  3307, 25044,  ...,  3702,  1422, 16667],
        [ 7133,  3345, 26632,  ..., 30329, 29646,  5542],
        ...,
        [ 5008,  2037, 28759,  ..., 21050, 26307, 22117],
        [ 2676, 19549, 19862,  ..., 19826, 24890, 21181],
        [24339, 30171,  2800,  ..., 16546, 23018, 17818]], device='cuda:0')
torch.Size([32, 128]) 

mask tensor([[1, 0, 0,  ..., 0, 1, 1],
        [0, 0, 1,  ..., 0, 1, 1],
        [1, 0, 1,  ..., 1, 0, 1],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 1],
        [0, 1, 1,  ..., 0, 0, 1]], device='cuda:0')
torch.Size([32, 128]) 

out tensor([-0.0916, -1.1550,  0.5840,  0.2825,  0.4304, -0.0398, -0.7031,  0.0298,
        -0.1126,  0.3433, -0.8002,  0.2898,  0.4382,  0.0493,  1.2940,  0.6615,
         0.1650, -0.1590,  0.6424, -0.4779, -0.3839,  0.6103, -0.1898,  0.4600,
         0.0261,  1.0196,  0.9436,  1.1912,  0.5440,  0.1727, -0.7015,  0.3988],
    

### Training loop with evaluation at end of each epoch



***TODO: Complete the training [15 points] and evaluation [10 points] loop below. Comments have been left to help you.***

In [21]:
# track loss
total_loss = 0.

# track time
total_time = 0.

# after log_freq batches, print loss and time
log_freq = 20

for e in range(epochs): # epoch loop

    print(f'Epoch {e+1}:')

    trf.train() # put model in training mode

    # iterate over training data in batches
    for idx, batch in enumerate(train_loader):
        start = time()
        # TODO: unpack batch
        texts, labels = batch
        # TODO: call tokenizer on entire batch at once
        encoded = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=trf.max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        labels = labels.float().to(device)
        # TODO: reset any gradients from previous batch to 0
        optimizer.zero_grad()
        # TODO: pass through transformer (you can call .to(device) on inputs)
        outputs = trf(encoded)
        # TODO: compute the loss between the outputs and the labels
        # labels must be floats and on the same device as the outputs
        loss = criterion(outputs, labels)
        # TODO: backpropagation
        loss.backward()
        # TODO: update model parameters
        optimizer.step()
        # TODO: track loss
        this_loss = loss.item()
        total_loss += this_loss
        # TODO: track time
        this_time = time() - start
        total_time += this_time
        # TODO: calculate average loss and time
        avg_loss = total_loss / (idx + 1)
        avg_time = total_time / (idx + 1)

        # log message
        if (idx+1) % log_freq == 0:
            msg =  f'\tBatch: {idx+1:04}, '
            msg += f'Loss: {this_loss:.4f}, Avg Loss: {avg_loss:.4f}, '
            msg += f'Time: {this_time:.4f}, Avg Time: {avg_time:.4f}'
            print(msg)

    # reset logging every epoch
    total_loss = 0.
    total_time = 0.

    # predictions over test set
    test_preds = []

    # true labels over test set
    test_labels = []

    # turn off gradient computation to save time - we are not updating here
    with torch.no_grad():

        print(f'Evaluating after epoch {e+1}:')

        # this line is known to cause problems with transformer encoders
        # depending on torch version...
        # trf.eval() # put model in evaluation mode

        # iterate over test set
        for idx, batch in enumerate(test_loader):

            # TODO: unpack batch
            texts, labels = batch
            # TODO: tokenize entire batch
            encoded = tokenizer(
                list(texts),
                padding=True,
                truncation=True,
                max_length=trf.max_length,
                return_tensors='pt'
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            labels = labels.to(device)
            # TODO: pass through transformer
            outputs = trf(encoded)
            # TODO: make predictions based on model outputs
            preds = torch.sigmoid(outputs)
            preds = (preds > 0.5).long()
            # TODO: store the corresponding labels
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())
    # TODO: evaluation - print a classification report
    print(classification_report(test_labels, test_preds, digits=4))

Epoch 1:
	Batch: 0020, Loss: 0.6714, Avg Loss: 0.7327, Time: 0.0610, Avg Time: 0.0860
	Batch: 0040, Loss: 0.6533, Avg Loss: 0.7088, Time: 0.0482, Avg Time: 0.0779
	Batch: 0060, Loss: 0.7211, Avg Loss: 0.6894, Time: 0.0416, Avg Time: 0.0683
	Batch: 0080, Loss: 0.5887, Avg Loss: 0.6795, Time: 0.0493, Avg Time: 0.0633
	Batch: 0100, Loss: 0.6526, Avg Loss: 0.6733, Time: 0.0468, Avg Time: 0.0601
	Batch: 0120, Loss: 0.6295, Avg Loss: 0.6674, Time: 0.0447, Avg Time: 0.0581
	Batch: 0140, Loss: 0.5124, Avg Loss: 0.6652, Time: 0.0548, Avg Time: 0.0564
	Batch: 0160, Loss: 0.5897, Avg Loss: 0.6613, Time: 0.0441, Avg Time: 0.0553
	Batch: 0180, Loss: 0.5628, Avg Loss: 0.6558, Time: 0.0474, Avg Time: 0.0543
	Batch: 0200, Loss: 0.6206, Avg Loss: 0.6502, Time: 0.0452, Avg Time: 0.0535
	Batch: 0220, Loss: 0.5374, Avg Loss: 0.6442, Time: 0.0723, Avg Time: 0.0528
	Batch: 0240, Loss: 0.5636, Avg Loss: 0.6377, Time: 0.0492, Avg Time: 0.0523
	Batch: 0260, Loss: 0.5303, Avg Loss: 0.6366, Time: 0.0713, Avg Tim

## Conceptual Questions

***1. What are the limitations of RNNs and CNNs that transformers aim to
solve? [5 points]***


Some limitations of RNNs includes poor parallelization whereby tokens cant be processed in parallel and vanishing or exploding gradients that makes it difficult to capture long-term dependencies. As for CNNS, they have a fixed context size and struggle to model relationships between distant tokens. Transformers solve these problems by being fully parallelizable due to self attention's ability to process all the tokens at once in parallel, capturing long range dependencies as each token attends to all others in the same layer, allowing the model to learn global relationships. Without a recurrent structure, there will be no vanishing/exploding gradients and gradients flow through attention layers more easily and more stable.

***2. Why do transformers require positional encoding, and how is it implemented? [5 points]***

Transformers require positional encoding because of how self-attention works and the need for sequential awareness. Self-attention only cares about the token relationships and not their position. However, to us, position of the tokens do matter. For instance, a dog chasing a cat is different from a cat chasing a dog. Yet, the self attention will see it as almost identical. Hence, positional encoding is required to give each token a sense of position such that the model can distinguish who did what and when. This is implemented by having a positional encoding added to each word embedding and this encoding will be unique for each position in the sentence. With such encoding, the model is able to differentiate between different permutations of the sentence.

***3. How does cross-attention differ from self-attention in the decoder? [5 points]***

Cross attention in the decoder allows the decoder to look at the encoder's output representations when generating the next token. The Q comes from the decoder's current hidden states and the K and V comes from the encoder's final outputs.

On the other hand, self-attention in the decoder allows the decoder to understand the relationships within the partially generated output. The Q, K and V all comes from the decoder's own previous outputs and a casual mask is applied so each token can only attend to earlier tokens and not any future ones.